# Feature Engineering — Preparação dos Dados para Modelagem

Este notebook transforma os dados limpos extraídos do BigQuery em features prontas
para o treinamento do modelo de classificação de churn.

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import sys
sys.path.append('../src')
from bigquery_client import run_query


df = run_query("""
    SELECT *
    FROM `churn_dataset.customers_cleaned`
""")

c:\Users\Augusto\Desktop\Projetos\churn-analysis\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 1. Encoding das Variáveis Categóricas

As variáveis booleanas já chegam como `True/False` do BigQuery e serão convertidas
para `0/1`. As demais categóricas receberão One-Hot Encoding via `pd.get_dummies`.

In [2]:
X = df.drop(columns=['customerID', 'Churn', 'tenure_group', 'TotalCharges', 'AvgMonthlySpend'])
y = df['Churn'].astype(int)

bool_cols = X.select_dtypes(include='boolean').columns
X[bool_cols] = X[bool_cols].astype(int)

colunas_categoricas = X.select_dtypes(include='object').columns.tolist()
colunas_numericas = X.select_dtypes(include='number').columns.tolist()

X = pd.get_dummies(X, columns=colunas_categoricas, drop_first=True)

print(f"Dimensões antes do encoding: {df.shape}")
print(f"Dimensões depois do encoding: {X.shape}")

Dimensões antes do encoding: (7043, 23)
Dimensões depois do encoding: (7043, 22)


C:\Users\Augusto\AppData\Local\Temp\ipykernel_11380\2135023004.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_categoricas = X.select_dtypes(include='object').columns.tolist()


## 2. Divisão em Treino e Teste

Os dados são divididos em treino (80%) e teste (20%), preservando a proporção
de classes com `stratify` para garantir que o desbalanceamento seja mantido
igualmente em ambos os conjuntos.

In [3]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify= y)

print(f"Tamanho do treino: {X_treino.shape[0]} amostras")
print(f"Tamanho do teste: {X_teste.shape[0]} amostras")

print("\n--- Proporção de classes no treino ---")
print(y_treino.value_counts(normalize = True))

print("\n--- Proporção de classes no teste ---")
print(y_teste.value_counts(normalize = True))

Tamanho do treino: 5634 amostras
Tamanho do teste: 1409 amostras

--- Proporção de classes no treino ---
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

--- Proporção de classes no teste ---
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


## 3. Modelo Baseline — Regressão Logística

Treinado primeiro sem balanceamento para estabelecer uma linha de base e entender
o comportamento do modelo com os dados originais.

In [4]:
colunas_para_escalar = ['tenure', 'MonthlyCharges']

scaler = StandardScaler()

X_treino_escalado = X_treino.copy()
X_teste_escalado = X_teste.copy()

X_treino_escalado[colunas_para_escalar] = scaler.fit_transform(X_treino[colunas_para_escalar])
X_teste_escalado[colunas_para_escalar] = scaler.transform(X_teste[colunas_para_escalar])

modelo_log = LogisticRegression(max_iter=1000, random_state=42)
modelo_log.fit(X_treino_escalado, y_treino)

y_pred_log = modelo_log.predict(X_teste_escalado)
y_proba_log = modelo_log.predict_proba(X_teste_escalado)[:, 1]

print("Modelo treinado com sucesso.")
print(f"Número de coeficientes estimados: {len(modelo_log.coef_[0])}")

Modelo treinado com sucesso.
Número de coeficientes estimados: 22


### 3.1. Avaliação do Modelo Baseline

Avaliando o desempenho inicial da Regressão Logística sem balanceamento, usando
múltiplas métricas — a acurácia isoladamente é enganosa em datasets desbalanceados.

In [5]:
print('--- Métricas do Modelo: Regressão Logística ---')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_log):.3f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_log):.3f}')
print(f'Recall:    {recall_score(y_teste, y_pred_log):.3f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred_log):.3f}')
print(f'AUC-ROC:   {roc_auc_score(y_teste, y_proba_log):.3f}')

print('\n--- Matriz de Confusão ---')
print(confusion_matrix(y_teste, y_pred_log))

print('\n--- Relatório de Classificação Completo ---')
print(classification_report(y_teste, y_pred_log, target_names=['Ativo', 'Churn']))

--- Métricas do Modelo: Regressão Logística ---
Acurácia:  0.793
Precisão:  0.635
Recall:    0.516
F1-Score:  0.569
AUC-ROC:   0.828

--- Matriz de Confusão ---
[[924 111]
 [181 193]]

--- Relatório de Classificação Completo ---
              precision    recall  f1-score   support

       Ativo       0.84      0.89      0.86      1035
       Churn       0.63      0.52      0.57       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.79      1409



O modelo baseline apresenta **AUC-ROC de 0.828** — boa capacidade de separação entre
as classes — mas o **recall de churn de apenas 51.6%** é crítico para o negócio: 181
clientes que cancelariam passaram despercebidos. Isso é esperado em dados desbalanceados
com threshold padrão de 0.5. O SMOTE deve melhorar o recall sacrificando um pouco
a precisão — trade-off desejado no contexto de retenção de clientes.

## 4. Balanceamento de Classes com SMOTE

Aplicado apenas no conjunto de treino para evitar data leakage — o teste permanece
com a distribuição original para uma avaliação realista.

In [6]:
smote = SMOTE(random_state=42)
X_treino_bal, y_treino_bal = smote.fit_resample(X_treino_escalado, y_treino)

print('--- Proporção de classes ANTES do SMOTE ---')
print(y_treino.value_counts())

print('\n--- Proporção de classes DEPOIS do SMOTE ---')
print(y_treino_bal.value_counts())

--- Proporção de classes ANTES do SMOTE ---
Churn
0    4139
1    1495
Name: count, dtype: int64

--- Proporção de classes DEPOIS do SMOTE ---
Churn
0    4139
1    4139
Name: count, dtype: int64


### 4.1. Retreinamento com Dados Balanceados

In [7]:
modelo_log_smote = LogisticRegression(max_iter=1000, random_state=42)
modelo_log_smote.fit(X_treino_bal, y_treino_bal)

y_pred_log_smote = modelo_log_smote.predict(X_teste_escalado)
y_proba_log_smote = modelo_log_smote.predict_proba(X_teste_escalado)[:, 1]

print('--- Métricas do Modelo: Regressão Logística + SMOTE ---')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_log_smote):.3f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_log_smote):.3f}')
print(f'Recall:    {recall_score(y_teste, y_pred_log_smote):.3f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred_log_smote):.3f}')
print(f'AUC-ROC:   {roc_auc_score(y_teste, y_proba_log_smote):.3f}')

print('\n--- Matriz de Confusão ---')
print(confusion_matrix(y_teste, y_pred_log_smote))

print('\n--- Relatório de Classificação Completo ---')
print(classification_report(y_teste, y_pred_log_smote, target_names=['Ativo', 'Churn']))

--- Métricas do Modelo: Regressão Logística + SMOTE ---
Acurácia:  0.749
Precisão:  0.521
Recall:    0.706
F1-Score:  0.599
AUC-ROC:   0.820

--- Matriz de Confusão ---
[[792 243]
 [110 264]]

--- Relatório de Classificação Completo ---
              precision    recall  f1-score   support

       Ativo       0.88      0.77      0.82      1035
       Churn       0.52      0.71      0.60       374

    accuracy                           0.75      1409
   macro avg       0.70      0.74      0.71      1409
weighted avg       0.78      0.75      0.76      1409



O SMOTE trouxe a melhora esperada no recall de churn — de **51.6% para 70.6%** —
identificando agora 264 dos 374 churns reais, contra 193 antes. O trade-off foi
a queda na precisão (de 63.5% para 52.1%) e na acurácia (de 79.3% para 74.9%),
ambas esperadas e aceitáveis no contexto de retenção — é preferível abordar alguns
falsos alarmes a deixar passar clientes que realmente vão cancelar.

A AUC-ROC se manteve estável (0.820), confirmando que a capacidade de separação
do modelo não foi prejudicada pelo balanceamento.

## 5. Modelo Random Forest + SMOTE

In [16]:
modelo_rf = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_rf.fit(X_treino_bal, y_treino_bal)

y_pred_rf = modelo_rf.predict(X_teste_escalado)
y_proba_rf = modelo_rf.predict_proba(X_teste_escalado)[:, 1]

print('--- Métricas do Modelo: Random Forest + SMOTE ---')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_rf):.3f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_rf):.3f}')
print(f'Recall:    {recall_score(y_teste, y_pred_rf):.3f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred_rf):.3f}')
print(f'AUC-ROC:   {roc_auc_score(y_teste, y_proba_rf):.3f}')

print('\n--- Matriz de Confusão ---')
print(confusion_matrix(y_teste, y_pred_rf))

print('\n--- Relatório de Classificação Completo ---')
print(classification_report(y_teste, y_pred_rf, target_names=['Ativo', 'Churn']))

--- Métricas do Modelo: Random Forest + SMOTE ---
Acurácia:  0.762
Precisão:  0.546
Recall:    0.615
F1-Score:  0.579
AUC-ROC:   0.797

--- Matriz de Confusão ---
[[844 191]
 [144 230]]

--- Relatório de Classificação Completo ---
              precision    recall  f1-score   support

       Ativo       0.85      0.82      0.83      1035
       Churn       0.55      0.61      0.58       374

    accuracy                           0.76      1409
   macro avg       0.70      0.72      0.71      1409
weighted avg       0.77      0.76      0.77      1409



O Random Forest + SMOTE apresentou recall de **61.5%** e AUC-ROC de **0.797** —
desempenho similar ao XGBoost padrão mas ainda abaixo da Regressão Logística com
SMOTE em recall (70.6%), a métrica mais crítica para o negócio de retenção.

## 6. Modelo XGBoost

O XGBoost é treinado sobre os dados balanceados pelo SMOTE para comparar seu
poder preditivo com a Regressão Logística. Por ser baseado em árvores de decisão,
não é sensível a escala — mas usamos os dados escalados para manter consistência.

In [8]:
modelo_xgb = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
modelo_xgb.fit(X_treino_bal, y_treino_bal)

y_pred_xgb = modelo_xgb.predict(X_teste_escalado)
y_proba_xgb = modelo_xgb.predict_proba(X_teste_escalado)[:, 1]

print('--- Métricas do Modelo: XGBoost + SMOTE ---')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_xgb):.3f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_xgb):.3f}')
print(f'Recall:    {recall_score(y_teste, y_pred_xgb):.3f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred_xgb):.3f}')
print(f'AUC-ROC:   {roc_auc_score(y_teste, y_proba_xgb):.3f}')

print('\n--- Matriz de Confusão ---')
print(confusion_matrix(y_teste, y_pred_xgb))

print('\n--- Relatório de Classificação Completo ---')
print(classification_report(y_teste, y_pred_xgb, target_names=['Ativo', 'Churn']))

--- Métricas do Modelo: XGBoost + SMOTE ---
Acurácia:  0.751
Precisão:  0.526
Recall:    0.612
F1-Score:  0.566
AUC-ROC:   0.785

--- Matriz de Confusão ---
[[829 206]
 [145 229]]

--- Relatório de Classificação Completo ---
              precision    recall  f1-score   support

       Ativo       0.85      0.80      0.83      1035
       Churn       0.53      0.61      0.57       374

    accuracy                           0.75      1409
   macro avg       0.69      0.71      0.70      1409
weighted avg       0.76      0.75      0.76      1409



## 6.1. Otimização de Hiperparâmetros — RandomizedSearchCV

Antes de concluir que a Regressão Logística é superior, otimizamos os hiperparâmetros
do XGBoost — ele tem muito mais parâmetros para ajustar e pode estar subperformando
com as configurações padrão.

In [13]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0]
}

xgb_random = RandomizedSearchCV(
    estimator= XGBClassifier(random_state = 42, eval_metric = 'logloss'),
    param_distributions = param_grid,
    n_iter = 50,
    scoring = 'roc_auc',
    cv = 5,
    random_state = 45,
    n_jobs = -1,
    verbose = 1
)

xgb_random.fit(X_treino_bal, y_treino_bal)
print(f'Melhores hiperparâmetros: {xgb_random.best_params_}')
print(f'Melhor AUC-ROC (cross-validation): {xgb_random.best_score_:.3f}')


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Melhores hiperparâmetros: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
Melhor AUC-ROC (cross-validation): 0.920


In [11]:
modelo_xgb_otimizado = xgb_random.best_estimator_

y_pred_xgb_ot = modelo_xgb_otimizado.predict(X_teste_escalado)
y_proba_xgb_ot = modelo_xgb_otimizado.predict_proba(X_teste_escalado)[:, 1]

print('--- Métricas do Modelo: XGBoost Otimizado + SMOTE ---')
print(f'Acurácia:  {accuracy_score(y_teste, y_pred_xgb_ot):.3f}')
print(f'Precisão:  {precision_score(y_teste, y_pred_xgb_ot):.3f}')
print(f'Recall:    {recall_score(y_teste, y_pred_xgb_ot):.3f}')
print(f'F1-Score:  {f1_score(y_teste, y_pred_xgb_ot):.3f}')
print(f'AUC-ROC:   {roc_auc_score(y_teste, y_proba_xgb_ot):.3f}')

print('\n--- Matriz de Confusão ---')
print(confusion_matrix(y_teste, y_pred_xgb_ot))

print('\n--- Relatório de Classificação Completo ---')
print(classification_report(y_teste, y_pred_xgb_ot, target_names=['Ativo', 'Churn']))

--- Métricas do Modelo: XGBoost Otimizado + SMOTE ---
Acurácia:  0.751
Precisão:  0.527
Recall:    0.610
F1-Score:  0.565
AUC-ROC:   0.798

--- Matriz de Confusão ---
[[830 205]
 [146 228]]

--- Relatório de Classificação Completo ---
              precision    recall  f1-score   support

       Ativo       0.85      0.80      0.83      1035
       Churn       0.53      0.61      0.57       374

    accuracy                           0.75      1409
   macro avg       0.69      0.71      0.70      1409
weighted avg       0.76      0.75      0.76      1409



O RandomizedSearchCV com 50 iterações encontrou os hiperparâmetros ótimos com
AUC-ROC de **0.920 na validação cruzada** — resultado promissor. Porém ao avaliar
no conjunto de teste, o XGBoost otimizado (0.798) apresentou ganho marginal em
relação ao padrão (0.785), e o recall de churn permaneceu praticamente igual
(61.0% vs 61.2%). Testar 100 iterações no RandomizedSearchCV não trouxe ganho
adicional, confirmando que 50 já foram suficientes para explorar o espaço de
hiperparâmetros desse problema.

## 7. Interpretação do Modelo — Odds Ratio

A Regressão Logística permite interpretar seus coeficientes através do Odds Ratio
(razão de chances) — entendendo como cada feature afeta a probabilidade de churn.

In [18]:
coeficientes = pd.DataFrame({
    'variavel': X_treino_bal.columns,
    'coeficiente': modelo_log_smote.coef_[0]
})

coeficientes['razao_de_chances'] = np.exp(coeficientes['coeficiente'])
coeficientes = coeficientes.sort_values('coeficiente', ascending=False)

print('--- Top 10 variáveis que MAIS AUMENTAM a probabilidade de churn ---')
print(coeficientes.head(10)[['variavel', 'razao_de_chances']].to_string(index=False))

print('--- Variáveis que REDUZEM a probabilidade de churn (OR < 1) ---')
print(coeficientes[coeficientes['razao_de_chances'] < 1][['variavel', 'razao_de_chances']].to_string(index=False))

--- Top 10 variáveis que MAIS AUMENTAM a probabilidade de churn ---
                      variavel  razao_de_chances
   InternetService_Fiber optic        186.158902
                  PhoneService         15.749591
               StreamingTV_Yes          7.978884
           StreamingMovies_Yes          7.868722
             MultipleLines_Yes          3.784470
PaymentMethod_Electronic check          2.577363
              OnlineBackup_Yes          2.554307
          DeviceProtection_Yes          2.058772
               TechSupport_Yes          1.878541
            OnlineSecurity_Yes          1.826422
--- Variáveis que REDUZEM a probabilidade de churn (OR < 1) ---
          variavel  razao_de_chances
     SeniorCitizen          0.964648
           Partner          0.865990
 Contract_One year          0.663104
        Dependents          0.659533
 Contract_Two year          0.370672
            tenure          0.363419
InternetService_No          0.009814
    MonthlyCharges          0.006

#### Interpretação — Odds Ratio

**Variáveis que mais aumentam a probabilidade de churn:**

- **InternetService_Fiber optic (186x):** valor extremamente alto — clientes de fibra ótica têm chance muito maior de cancelar, confirmando o sinal de alerta identificado na EDA
- **PhoneService (15.7x):** clientes com serviço telefônico têm chance significativamente maior de churn
- **StreamingTV e StreamingMovies (~8x):** serviços de streaming estão fortemente associados ao cancelamento — possivelmente clientes que assinam esses serviços encontram alternativas mais baratas no mercado
- **MultipleLines (3.8x):** múltiplas linhas aumentam a chance de churn
- **PaymentMethod_Electronic check (2.6x):** confirma o insight da EDA — esse método de pagamento está associado a maior churn

**Variáveis que mais reduzem a probabilidade de churn:**

- **MonthlyCharges (0.006):** surpreendente — após controlar as demais variáveis, mensalidade mais alta está associada a menor churn
- **InternetService_No (0.009):** clientes sem internet têm chance muito menor de cancelar
- **tenure (0.36):** quanto mais tempo o cliente permanece, menor a chance de churn — confirma o insight da EDA
- **Contract_Two year (0.37):** contratos bianuais reduzem fortemente o churn — alta barreira de saída
- **Contract_One year (0.66):** contratos anuais também reduzem, mas menos que os bianuais